In [1]:
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from PIL import Image
from torch.optim import Adam, SGD, lr_scheduler
import matplotlib.pyplot as plt
from torchvision import transforms, models
from UNET_LIB.InceptionUnet import InceptionUNet
# from unet import UNet2D
from utils import DiceLoss, SquarePad, SquarePad255
from Brats_Dataset import Brats_Dataset
from torch.utils.data import Dataset, Subset, DataLoader

train_total=10240

group_spec={
    'perfe':160,
    'poly+':213,
    'poly-':572,
    'rough':868,
    'bbox_msk':1562,
    'sam_box':1562,
}

mult_spec={
    'perfe':[1,2,4,8,16,32,64],
    'poly+':[1,2,4,8,16,32,train_total/group_spec['poly+']],
    'poly-':[1,2,4,8,16,train_total/group_spec['poly-']],
    'rough':[1,2,4,8,train_total/group_spec['rough']],
    'bbox_msk':[1,2,4,train_total/group_spec['bbox_msk']],
    'sam_box':[1,2,4,train_total/group_spec['sam_box']]
}

epochs0 = 80
pretrained = 'pb'
assert pretrained in ['pb','fb','ub']


img_preprocess = transforms.Compose([    
#     SquarePad(),
#     transforms.Resize(512),
    transforms.RandomEqualize(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

val_preprocess = transforms.Compose([    
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])


msk_preprocess = transforms.Compose([
])

In [3]:
label_type='sam_box'
assert label_type in ['perfe','poly+','poly-','rough','bbox_msk','sam_box']
group_size = group_spec[label_type]
DATA = Brats_Dataset('/data/Brats20/','trainval',label_type,'f_', img_preprocess, msk_preprocess,crop=False)
num_classes = 2

loss_fn = nn.CrossEntropyLoss(ignore_index=255) 
for multiplicity in reversed(mult_spec[label_type]):
    epochs = int(epochs0*6/multiplicity)

    model = InceptionUNet(1, n_classes=num_classes)

    
    
    
    model = nn.DataParallel(model).cuda()
    #short for DeepLab-VOC-{label_type}-m{multiplicity}
    model_name = f'XUNet_B{label_type}_m{round(multiplicity)}_{pretrained}' 
    
    train_subset = Subset(DATA, list(range(round((group_size*multiplicity)))))
    val_subset = Subset(DATA, list(range(train_total, len(DATA))))
    train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, drop_last=True, num_workers=8)
    val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, drop_last=False, num_workers=8)
    
    min_loss = np.inf
    fin_epoch = 0
    
    optimizer = Adam(model.parameters(),lr=1e-3,eps=0.1, weight_decay=1e-6)
    
    for e in pbar:
        train_loss, val_loss = 0, 0
        
        model.train()
        for idx,(X,y) in enumerate(train_loader):

            with torch.no_grad():
                valid_loc = (y != 255).cuda()
                y = y.squeeze().cuda()*valid_loc
            optimizer.zero_grad()
            
            yhat = model(X.contiguous().cuda())
            loss = (loss_fn(yhat,y)*valid_loc).mean()
            loss.backward()
            optimizer.step()

            
            train_loss += loss.item()
        train_loss /= (idx+1)

        model.eval()
        with torch.no_grad():
            class_intersect = np.zeros((num_classes,),dtype='float')
            class_union= np.zeros((num_classes,),dtype='float')
            for idx,(X,y) in enumerate(val_loader):
                y = y.cuda().contiguous().flatten()
                
                yhat = model(X.contiguous().cuda())
                yhat_lab = torch.argmax(yhat, dim=1).flatten()
                yhat_lab[y == 255] = 255

                for j in range(num_classes):

                    y_bi = y == j
                    yhat_bi = yhat_lab == j
                    I = ((y_bi * yhat_bi).sum()).item()
                    U = (y_bi.sum() + yhat_bi.sum() - I).item()
                    assert I <= U
                    class_intersect[j] += I
                    class_union[j] += U
            
            IOUs = class_intersect/class_union
            val_loss=-np.mean(IOUs)
            pbar.set_description(f'Train loss: {train_loss}| IOU: {-val_loss}')
            
            #Dynamic class weight adjustment for loss function
            boost = 1/np.clip(IOUs,5e-2,1)
            boost = torch.tensor(boost/boost.sum(),dtype=torch.float).cuda()

        loss_fn = nn.CrossEntropyLoss(boost, reduction='none')   

        if e <10:
            continue

        if val_loss < min_loss:
            min_loss = val_loss
            to_save ={
                'min_loss': min_loss,
                'fin_epoch': e+fin_epoch+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict()}
            if pretrained=='fb':
                to_save['scheduler_state_dict']= scheduler.state_dict()
                
            torch.save(to_save, f'/data/model_checkpoints/{model_name}.pth')  


  0%|                                                                                                  | 0/80 [00:00<?, ?it/s]

new model training XUNet_Brough_m12_pb


Train loss: 0.006371136124653276| IOU: 0.8296967839975902: 100%|███████████████████████████| 80/80 [4:47:34<00:00, 215.68s/it]
  0%|                                                                                                  | 0/80 [00:00<?, ?it/s]

new model training XUNet_Brough_m8_pb


Train loss: 0.0062000496175638| IOU: 0.8046601235406767: 100%|█████████████████████████████| 80/80 [3:21:16<00:00, 150.95s/it]
  0%|                                                                                                  | 0/80 [00:00<?, ?it/s]

...

Train loss: 0.006729340305217125| IOU: 0.8023164508397673: 100%|████████████████████████████| 80/80 [1:50:21<00:00, 82.77s/it]
  0%|                                                                                                  | 0/80 [00:00<?, ?it/s]

new model training XUNet_Brough_m2_pb


Train loss: 0.007902819641727817| IOU: 0.7820414770939372: 100%|████████████████████████████| 80/80 [1:04:35<00:00, 48.44s/it]
  0%|                                                                                                  | 0/80 [00:00<?, ?it/s]

new model training XUNet_Brough_m1_pb


Train loss: 0.010009922949528252| IOU: 0.7456593943558133: 100%|██████████████████████████████| 80/80 [42:02<00:00, 31.53s/it]


In [3]:
pretrained='pb'
performance={}
num_classes=2
test_preprocess = transforms.Compose([    

    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

test_loader = DataLoader( Brats_Dataset('/data/Brats20/','test','perfe',
                                              'f_', test_preprocess, msk_preprocess,crop=False), 
                         batch_size=64, shuffle=False, num_workers=8)

model = InceptionUNet(1, n_classes=num_classes)
model = nn.DataParallel(model).cuda()

for label_type in mult_spec.keys():
    performance[label_type]={}
    
    for multiplicity in mult_spec[label_type]:
        
        model_name = f'XUNet_B{label_type}_m{round(multiplicity)}_{pretrained}' 
        checkpoint = torch.load(f'/data/model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        
        model.eval()
        print(f'Working on label_type={label_type}:{round(multiplicity)}')
        class_intersect = np.zeros((num_classes, ),dtype='float')
        class_union = np.zeros((num_classes, ),dtype='float')
    
        with torch.no_grad():
            for idx,(X,y) in enumerate(test_loader):
                y = y.flatten()

                yhat = model(X.contiguous().cuda())
                yhat_lab = torch.argmax(yhat.cpu(), dim=1).flatten()
                skip_id = np.argwhere(y == 255)
                yhat_lab[skip_id] = 255

                for j in range(num_classes):

                    y_bi = y == j
                    yhat_bi = yhat_lab == j
                    I = ((y_bi * yhat_bi).sum()).item()
                    U = (y_bi.sum() + yhat_bi.sum() - I).item()
                    assert I <= U
                    class_intersect[j] += I
                    class_union[j] += U
                    
        performance[label_type][multiplicity]=(class_intersect, class_union)
        
np.save(f'Brats_Inter-Union_{pretrained}', performance)


Working on label_type=perfe:1
Working on label_type=perfe:2
Working on label_type=perfe:4
Working on label_type=perfe:8
Working on label_type=perfe:16
Working on label_type=perfe:32
Working on label_type=perfe:64
Working on label_type=poly+:1
Working on label_type=poly+:2
Working on label_type=poly+:4
Working on label_type=poly+:8
Working on label_type=poly+:16
Working on label_type=poly+:32
Working on label_type=poly+:48
Working on label_type=poly-:1
Working on label_type=poly-:2
Working on label_type=poly-:4
Working on label_type=poly-:8
Working on label_type=poly-:16
Working on label_type=poly-:18
Working on label_type=rough:1
Working on label_type=rough:2
Working on label_type=rough:4
Working on label_type=rough:8
Working on label_type=rough:12
Working on label_type=bbox_msk:1
Working on label_type=bbox_msk:2
Working on label_type=bbox_msk:4
Working on label_type=bbox_msk:7


In [9]:
####some old code --abandoned
for i in range(len(multiplicity_list)): 
    print(f'multiplicity:{multiplicity_list[i]}: ', np.mean(class_intersect[i]/class_union[i]),\
          np.std(class_intersect[i]/class_union[i]),
          np.mean(2*class_intersect[i]/(class_intersect[i]+class_union[i])),
          np.std(2*class_intersect[i]/(class_intersect[i]+class_union[i])))

multiplicity:1:  0.13222235168842367 0.07390113808500424 0.2262246641991838 0.11276045202718044
multiplicity:2:  0.14710304129698945 0.06883423321089506 0.25004757290650065 0.10740431791076314
multiplicity:4:  0.15079518159805505 0.07190442744217519 0.2553373460144782 0.10831801359687405
multiplicity:8:  0.15772297539210886 0.0775480918617959 0.26475788700436587 0.1157308321814092


In [3]:
#snipping result

label_type='perfe'
test_loader = DataLoader(VOCSeg_Dataset('/data/VOC2012/','val','perfe',img_preprocess,msk_preprocess,crop=True,crop_size=500), batch_size=16, shuffle=False)

for multiplicity in mult_spec[label_type]:
        
    model = UNet(in_channels = 1, num_classes=num_classes)
    model = nn.DataParallel(model).cuda()
        
    model_name = f'DLab_V{label_type}_m{round(multiplicity)}_{pretrained}' 
    try:
        checkpoint = torch.load(f'/data/model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        print('Loaded', model_name)
    except:
        print(f'Error loading {model_name}')
        assert False

    model.eval()
    with torch.no_grad():    
        class_intersect = np.zeros((num_classes,),dtype='float')
        class_union= np.zeros((num_classes,),dtype='float')

        for idx,(X,y) in enumerate(test_loader):
            y = y.flatten()

            yhat = model(X.contiguous().cuda())['out']
            yhat_lab = torch.argmax(yhat.cpu(), dim=1).flatten()
            skip_id = np.argwhere(y == 255)
            yhat_lab[skip_id] = 255

            for j in range(num_classes):

                y_bi = y == j
                yhat_bi = yhat_lab == j
                I = (y_bi * yhat_bi).sum()
                U = y_bi.sum() + yhat_bi.sum() - I
                assert I <= U
                class_intersect[j] += I
                class_union[j] += U

        IOUs = class_intersect/class_union
        val_loss=-np.mean(IOUs)
        print('IOU', -val_loss)
            

Loaded DLab_Vperfe_m1_fb
IOU 0.18235766244660132
Loaded DLab_Vperfe_m2_fb
IOU 0.25649041047700427
Loaded DLab_Vperfe_m4_fb
IOU 0.39581744544167125
Loaded DLab_Vperfe_m8_fb
IOU 0.44429039939213066
Loaded DLab_Vperfe_m16_fb
IOU 0.5388698538183123
Loaded DLab_Vperfe_m32_fb
IOU 0.6126735413546928
Loaded DLab_Vperfe_m50_fb
IOU 0.6276882060092092
